In [1]:
import pandas as pd
import numpy as np
import json

In [2]:
sinmorphy_df = pd.read_csv("../data/century_morphology_data_v6.csv")
sinling_df = pd.read_csv("../data/century_morphology_data_sinling_full.csv")
tnt_df = pd.read_csv("../data/century_morphology_data_tnt_full.csv")

In [3]:
sinmorphy_df.head()

,century,sentence_index,word,new_lemma,morphology,pos,all_tags,all_lemmas,all_morphologies
0,5,0,කාම,කාම,කාම+N+NUTR+PL+NOM,NOUN,NOUN,කාම,කාම+N+NUTR+PL+NOM
1,5,0,ලොක,ලො,ලො+GUESS+N+F+SG+ID+ACC,NOUN,NOUN,ලො,ලො+GUESS+N+F+SG+ID+ACC
2,5,0,රූප,රූප,රූප+N+NUTR+PL+NOM,NOUN,NOUN,රූප,රූප+N+NUTR+PL+NOM
3,5,0,ලොක,ලො,ලො+GUESS+N+F+SG+ID+ACC,NOUN,NOUN,ලො,ලො+GUESS+N+F+SG+ID+ACC
4,5,0,අරූපලෝක,අරූපලෝ,අරූපලෝ+GUESS+N+F+SG+ID+ACC,NOUN,NOUN,අරූපලෝ,අරූපලෝ+GUESS+N+F+SG+ID+ACC


In [4]:
sinling_df.head()

,century,sentence_index,word,sinling_pos,sinling_upd_pos
0,5,0,කාම,UNK,UNKNOWN
1,5,0,ලොක,UNK,UNKNOWN
2,5,0,රූප,UNK,UNKNOWN
3,5,0,ලොක,UNK,UNKNOWN
4,5,0,අරූපලෝක,UNK,UNKNOWN


In [5]:
tnt_df.head()

,century,sentence_index,word,new_lemma,morphology,pos,all_tags,all_lemmas,all_morphologies,pos_tnt,tnt_upd_pos
0,5,0,කාම,කාම,කාම+N+NUTR+PL+NOM,NOUN,NOUN,කාම,කාම+N+NUTR+PL+NOM,JJ,ADJECTIVE
1,5,0,ලොක,ලො,ලො+GUESS+N+F+SG+ID+ACC,NOUN,NOUN,ලො,ලො+GUESS+N+F+SG+ID+ACC,UNK,UNKNOWN
2,5,0,රූප,රූප,රූප+N+NUTR+PL+NOM,NOUN,NOUN,රූප,රූප+N+NUTR+PL+NOM,NNC,NOUN
3,5,0,ලොක,ලො,ලො+GUESS+N+F+SG+ID+ACC,NOUN,NOUN,ලො,ලො+GUESS+N+F+SG+ID+ACC,UNK,UNKNOWN
4,5,0,අරූපලෝක,අරූපලෝ,අරූපලෝ+GUESS+N+F+SG+ID+ACC,NOUN,NOUN,අරූපලෝ,අරූපලෝ+GUESS+N+F+SG+ID+ACC,UNK,UNKNOWN


In [6]:
len(sinmorphy_df), len(sinling_df), len(tnt_df)

(64813, 64813, 64813)

In [7]:
sinmorphy_df["tnt_pos"] = tnt_df["tnt_upd_pos"]

In [8]:
sinmorphy_df["sinling_pos"] = sinling_df["sinling_upd_pos"]

In [9]:
sinmorphy_df.head()

,century,sentence_index,word,new_lemma,morphology,pos,all_tags,all_lemmas,all_morphologies,tnt_pos,sinling_pos
0,5,0,කාම,කාම,කාම+N+NUTR+PL+NOM,NOUN,NOUN,කාම,කාම+N+NUTR+PL+NOM,ADJECTIVE,UNKNOWN
1,5,0,ලොක,ලො,ලො+GUESS+N+F+SG+ID+ACC,NOUN,NOUN,ලො,ලො+GUESS+N+F+SG+ID+ACC,UNKNOWN,UNKNOWN
2,5,0,රූප,රූප,රූප+N+NUTR+PL+NOM,NOUN,NOUN,රූප,රූප+N+NUTR+PL+NOM,NOUN,UNKNOWN
3,5,0,ලොක,ලො,ලො+GUESS+N+F+SG+ID+ACC,NOUN,NOUN,ලො,ලො+GUESS+N+F+SG+ID+ACC,UNKNOWN,UNKNOWN
4,5,0,අරූපලෝක,අරූපලෝ,අරූපලෝ+GUESS+N+F+SG+ID+ACC,NOUN,NOUN,අරූපලෝ,අරූපලෝ+GUESS+N+F+SG+ID+ACC,UNKNOWN,UNKNOWN


In [10]:
list(sinmorphy_df["all_tags"].value_counts().to_dict().keys())

['NOUN',
 'Proper noun or Invalid',
 'Verb',
 'PART',
 'NOUN | Verb',
 'PROPER NOUN',
 'NOUN | PART',
 'Verb | PART',
 'NOUN | Verb | PART',
 'ADJECTIVE',
 'PRONOUN',
 'NOUN | ADJECTIVE',
 'NOUN | Verb | ADJECTIVE',
 'Verb | ADJECTIVE',
 'FOREIGN WORD',
 'NOUN | PART | ADJECTIVE',
 'PART | ADJECTIVE',
 'PRONOUN | Verb']

In [22]:
def finalize_all_columns(row):
    # 1. Setup lists from the string columns
    all_tags_list = [t.strip() for t in str(row['all_tags']).split('|')]
    all_morphs_list = [m.strip() for m in str(row['all_morphologies']).split('|')]
    
    selected_index = 0
    
    # 2. Logic to find the correct index based on your priority rules
    if len(all_tags_list) > 1:
        tnt_res = row['tnt_pos']
        sinling_res = row['sinling_pos']
        
        if tnt_res != 'UNKNOWN' and tnt_res in all_tags_list:
            selected_index = all_tags_list.index(tnt_res)
        elif sinling_res != 'UNKNOWN' and sinling_res in all_tags_list:
            selected_index = all_tags_list.index(sinling_res)

    # 3. Extract final values
    final_tag = all_tags_list[selected_index]
    
    try:
        final_morph = all_morphs_list[selected_index]
    except IndexError:
        final_morph = all_morphs_list[0] if all_morphs_list else "UNKNOWN"

    # 4. Extract Lemma (e.g., from "කාම+N+NUTR+PL+NOM" get "කාම")
    # We split by '+' and take the first part
    if '+' in final_morph:
        final_lemma = final_morph.split('+')[0]
    else:
        # Fallback if the morphology format is different or missing '+'
        final_lemma = final_morph 

    return pd.Series([final_tag, final_morph, final_lemma])

In [23]:
sinmorphy_df[['final_pos', 'final_morphology', 'final_lemma']] = sinmorphy_df.apply(finalize_all_columns, axis=1)

In [24]:
sinmorphy_df['final_pos'].value_counts()

final_pos
NOUN                      31890
Verb                       9963
PART                       9329
Proper noun or Invalid     9143
PROPER NOUN                2264
ADJECTIVE                  1661
PRONOUN                     546
FOREIGN WORD                 17
Name: count, dtype: int64

In [25]:
sinmorphy_df['final_morphology']

0                     කාම+N+NUTR+PL+NOM
1                ලො+GUESS+N+F+SG+ID+ACC
2                     රූප+N+NUTR+PL+NOM
3                ලො+GUESS+N+F+SG+ID+ACC
4            අරූපලෝ+GUESS+N+F+SG+ID+ACC
                      ...              
64808    වැඩු+GUESS+N+RT+NUTR+SG+DF+GEN
64809            හිරමන+N+NUTR+SG+DF+INT
64810         තම+GUESS+N+RT+M+SG+DF+NOM
64811                      ඇ+GUESS+N+RT
64812                       හා+PARTICLE
Name: final_morphology, Length: 64813, dtype: object

In [26]:
sinmorphy_df['final_lemma']

0           කාම
1            ලො
2           රූප
3            ලො
4        අරූපලෝ
          ...  
64808      වැඩු
64809     හිරමන
64810        තම
64811         ඇ
64812        හා
Name: final_lemma, Length: 64813, dtype: object